# Spatio-Temporal GNN: Biology Simulation Verification

This notebook verifies the mathematical and computational models for **Lentiviral Genomic Integration** and **Conformational Ligand Kinetics** designed for the `babelForge` Spatio-Temporal Graph Neural Network (ST-GNN) engine.

## Models Included:
1. **Ellipsoidal KNN Connectome Generator** (deterministic initialization)
2. **Retroviral Insertion & Node-Splitting Model** (spatial displacement and wiring splits)
3. **Insertional Mutagenesis Loop-Entropy Risk Model** (Shannon cycle entropy)
4. **Conformational Logic Gates** (pH and cluster density triggers modulating adjacency weights)

In [ ]:
import math
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple, Set
print("Libraries successfully imported. Ready to initialize models.")

## 1. Connectome Base Generator (Ellipsoidal KNN Backbone)

In [ ]:
class CellNode:
    def __init__(self, node_id: int, name: str, x: float, y: float, z: float, region: str, layer: str):
        self.id = node_id
        self.name = name
        self.type = "host"
        self.x = x
        self.y = y
        self.z = z
        self.region = region
        self.layer = layer
        self.conformation = "A"
        self.local_ph = 7.2
        self.mutation_score = 0.0

def generate_base_connectome(num_nodes: int = 50, seed: int = 4242) -> Tuple[List[CellNode], List[Tuple[int, int]]]:
    random.seed(seed)
    nodes = []
    edges = []
    
    # Ellipsoidal boundaries
    a, b, c = 40.0, 30.0, 45.0
    
    for i in range(num_nodes):
        while True:
            x = random.uniform(-a, a)
            y = random.uniform(-b, b)
            z = random.uniform(-c, c)
            if (x/a)**2 + (y/b)**2 + (z/c)**2 <= 1.0:
                break
        
        dist = math.sqrt(x*x + y*y + z*z)
        layer = "Deep" if dist < 18 else ("Subcortical" if dist < 28 else "Cortical")
        regions = ["Default", "Control", "Limbic", "Visual", "SomatoMotor", "VentAttn"]
        region = random.choice(regions)
        
        nodes.append(CellNode(i, f"Host_{i}", round(x, 2), round(y, 2), round(z, 2), region, layer))
    
    # KNN structural connections (K=2)
    for i in range(num_nodes):
        dists = []
        for j in range(num_nodes):
            if i == j: continue
            d = (nodes[i].x - nodes[j].x)**2 + (nodes[i].y - nodes[j].y)**2 + (nodes[i].z - nodes[j].z)**2
            dists.append((j, d))
        dists.sort(key=lambda x: x[1])
        
        for j, _ in dists[:2]:
            pair = tuple(sorted((i, j)))
            if pair not in edges:
                edges.append(pair)
                
    return nodes, edges

nodes, edges = generate_base_connectome(45)
print(f"Generated Connectome: {len(nodes)} nodes, {len(edges)} structural edges.")

## 2. Retroviral Genomic Integration (Node-Splitting)

In [ ]:
def integrate_retrovirus(
    base_nodes: List[CellNode], 
    base_edges: List[Tuple[int, int]], 
    dosage: float, 
    mutation_rate: float,
    seed: int = 1212
) -> Tuple[List[CellNode], List[Tuple[int, int, str]]]:
    random.seed(seed)
    nodes = [CellNode(n.id, n.name, n.x, n.y, n.z, n.region, n.layer) for n in base_nodes]
    
    # Standardize edge structures
    edges = [(u, v, "structural") for u, v in base_edges]
    
    num_splits = min(int(dosage * 2), int(len(nodes) * 0.3))
    if num_splits <= 0: 
        return nodes, edges
        
    # Split target selection (randomized)
    target_ids = random.sample([n.id for n in nodes], num_splits)
    next_id = max(n.id for n in nodes) + 1
    
    for target_id in target_ids:
        target_idx = next(idx for idx, n in enumerate(nodes) if n.id == target_id)
        target_node = nodes[target_idx]
        target_node.type = "mutated_site"
        target_node.mutation_score = min(1.0, target_node.mutation_score + 0.3)
        
        # Displacement
        epsilon = 3.0
        theta = random.uniform(0, math.pi * 2)
        phi = math.acos(random.uniform(-1, 1))
        dx = epsilon * math.sin(phi) * math.cos(theta)
        dy = epsilon * math.sin(phi) * math.sin(theta)
        dz = epsilon * math.cos(phi)
        
        # Create split daughter cell
        daughter = CellNode(
            next_id,
            f"{target_node.name}_daughter",
            round(target_node.x + dx, 2),
            round(target_node.y + dy, 2),
            round(target_node.z + dz, 2),
            target_node.region,
            target_node.layer
        )
        nodes.append(daughter)
        daughter_id = next_id
        next_id += 1
        
        # Create intervening viral insertion vector
        viral = CellNode(
            next_id,
            f"VIRAL_{target_node.id}",
            round((target_node.x + daughter.x) / 2, 2),
            round((target_node.y + daughter.y) / 2, 2),
            round((target_node.z + daughter.z) / 2, 2),
            target_node.region,
            target_node.layer
        )
        viral.type = "viral_vector"
        nodes.append(viral)
        viral_id = next_id
        next_id += 1
        
        # Rewire host-daughter edges
        for idx, (u, v, etype) in enumerate(edges):
            if etype == "structural":
                if u == target_id and random.random() > 0.5:
                    edges[idx] = (daughter_id, v, etype)
                elif v == target_id and random.random() > 0.5:
                    edges[idx] = (u, daughter_id, etype)
                    
        # Connect host <-> viral <-> daughter
        edges.append((target_node.id, viral_id, "viral_insertion"))
        edges.append((viral_id, daughter_id, "viral_insertion"))
        
    return nodes, edges

v_nodes, v_edges = integrate_retrovirus(nodes, edges, 2.5, 0.15)
print(f"Integrated retrovirus. Post-Sim count: {len(v_nodes)} nodes, {len(v_edges)} edges.")

## 3. Loop-Entropy Mutagenesis Risk Model

In [ ]:
def extract_loops(nodes: List[CellNode], edges: List[Tuple[int, int, str]]) -> List[List[int]]:
    G = nx.Graph()
    for n in nodes:
        G.add_node(n.id)
    for u, v, _ in edges:
        G.add_edge(u, v)
        
    # Find all simple cycles under cycle basis of length <= 4
    cycles = nx.cycle_basis(G)
    filtered_cycles = [c for c in cycles if len(c) >= 3 and len(c) <= 4]
    return filtered_cycles

def calculate_loop_entropy(nodes: List[CellNode], loops: List[List[int]]) -> float:
    if not loops:
        return 0.0
    participation = {n.id: 0 for n in nodes}
    total = 0
    for loop in loops:
        for nid in loop:
            if nid in participation:
                participation[nid] += 1
                total += 1
                
    if total == 0:
        return 0.0
        
    entropy = 0.0
    for nid, count in participation.items():
        if count > 0:
            p = count / total
            entropy -= p * math.log2(p)
    return round(entropy, 4)

def evaluate_mutagenesis_risk(current_loops: int, baseline_loops: int, entropy: float) -> float:
    if baseline_loops == 0:
        return 0.0
    broken = max(0, baseline_loops - current_loops)
    broken_ratio = broken / baseline_loops
    
    entropy_factor = min(2.5, 5.0 / (entropy + 1.0)) if entropy > 0 else 2.5
    risk = broken_ratio * 0.7 + (1.0 - math.exp(-entropy * 0.05)) * 0.3 * entropy_factor
    return round(min(1.0, max(0.0, risk)) * 100, 1)

base_loops = extract_loops(nodes, [(u, v, "structural") for u, v in edges])
sim_loops = extract_loops(v_nodes, v_edges)
entropy = calculate_loop_entropy(v_nodes, sim_loops)
risk = evaluate_mutagenesis_risk(len(sim_loops), len(base_loops), entropy)

print(f"Baseline Loops: {len(base_loops)} | Active Loops: {len(sim_loops)}")
print(f"Shannon Cycle Entropy: {entropy} bits")
print(f"Mutagenesis Risk Index: {risk}%")

## 4. Conformational Environmental Logic Gates

In [ ]:
def apply_conformational_kinetics(
    nodes: List[CellNode],
    edges: List[Tuple[int, int, str]],
    pH: float,
    clique_density: float,
    ph_threshold: float = 6.5,
    baseline_density: float = 0.20,
    coupling_multiplier: float = 4.0
) -> Tuple[List[CellNode], Dict[Tuple[int, int], float]]:
    # Environmental logic gate check
    gate_engaged = pH < ph_threshold and clique_density > baseline_density
    
    # Step conformations
    for n in nodes:
        if gate_engaged:
            if random.random() < 0.6:
                n.conformation = "B" # Shift active
        else:
            if random.random() < 0.2:
                n.conformation = "A" # Closed conformation
                
    # Re-weight adjacency matrix
    adjacency_weights = {}
    for u, v, etype in edges:
        src_node = next(n for n in nodes if n.id == u)
        
        weight = 1.0
        if src_node.conformation == "B":
            weight *= coupling_multiplier
            
        if src_node.type == "mutated_site":
            weight *= (1.0 - src_node.mutation_score * 0.4)
            
        adjacency_weights[(u, v)] = round(weight, 3)
        
    return nodes, adjacency_weights

# Engage active microenvironment (acidic & dense)
k_nodes, k_weights = apply_conformational_kinetics(v_nodes, v_edges, pH=6.2, clique_density=0.28)
active_b = sum(1 for n in k_nodes if n.conformation == "B")
print(f"Conformational State B occupancy: {active_b} / {len(k_nodes)} nodes ({(active_b/len(k_nodes)*100):.1f}%)")
print(f"Maximum coupling weight reached: {max(k_weights.values())}x")

## 5. Visualising Connectome Graph Transformations

In [ ]:
def draw_topology(nodes: List[CellNode], edges: List[Tuple[int, int, str]], weights: Dict[Tuple[int, int], float]):
    plt.figure(figsize=(10, 8))
    G = nx.DiGraph()
    
    pos = {}
    node_colors = []
    node_sizes = []
    
    # Standardized color specs
    for n in nodes:
        G.add_node(n.id)
        pos[n.id] = (n.x, n.y) # Project 3D coordinate bounds in 2D
        
        if n.type == "viral_vector":
            node_colors.append("#10b981") # Emerald green
            node_sizes.append(180)
        elif n.type == "mutated_site":
            node_colors.append("#ef4444") # Red
            node_sizes.append(160)
        elif n.conformation == "B":
            node_colors.append("#a855f7") # Glowing purple
            node_sizes.append(180)
        else:
            node_colors.append("#cbd5e1") # Standard host slate
            node_sizes.append(80)
            
    # Connect edges
    edge_colors = []
    widths = []
    for u, v, etype in edges:
        G.add_edge(u, v)
        w = weights.get((u, v), 1.0)
        widths.append(w * 0.8)
        
        if etype == "viral_insertion":
            edge_colors.append("#10b981")
        elif w > 2.5:
            edge_colors.append("#a855f7")
        else:
            edge_colors.append("#94a3b8")
            
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=node_sizes, alpha=0.95)
    nx.draw_networkx_edges(G, pos, edge_color=edge_colors, width=widths, alpha=0.6, arrows=False)
    
    plt.title(f"ST-GNN Connectome - Mutagenesis Risk: {risk}% | State B: {(active_b/len(k_nodes)*100):.1f}%", fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.show()

draw_topology(k_nodes, v_edges, k_weights)